Assignment 1:
Create a delta table for storing Product data and perform incremental insert / update

Steps Involved:

Step1: Read Product.csv as a dataframe and save it as a delta table called Product

Step2: Read Product_delta.csv as dataframe and perform an upsert transaction on product delta table

Step3: Verify that new records are inserted and existing records are updated in the delta table

In [0]:
# Initialize SparkSession
from pyspark.sql import SparkSession

In [0]:
spark = SparkSession.builder \
    .appName("ProductDeltaTable") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

Step1: Read Product.csv as a dataframe and save it as a delta table called Product

In [0]:
# Read Product.csv into a DataFrame
product_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("path/to/Product.csv") # Replace with the actual path to your Product.csv

In [0]:
# Save the DataFrame as a Delta table
product_df.write.format("delta").mode("overwrite").saveAsTable("Product")

In [0]:
print("Delta table 'Product' created successfully from Product.csv")

Step 2: Read Product_data.csv as a DataFrame and perform an upsert transaction on the Product Delta table.
This step utilizes the merge operation in Delta Lake, which allows for both inserts and updates in a single transaction.

In [0]:
from delta.tables import DeltaTable

# Read Product_data.csv into a DataFrame
product_data_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("path/to/Product_data.csv") # Replace with the actual path to your Product_data.csv

# Get the DeltaTable object for the 'Product' table
product_delta_table = DeltaTable.forName(spark, "Product")

# Perform the upsert (merge) operation
product_delta_table.alias("target") \
    .merge(
        product_data_df.alias("source"),
        "target.ProductID = source.ProductID" # Assuming ProductID is the unique key
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

print("Upsert operation completed on 'Product' Delta table.")

Step 3: Verify that new records are inserted and existing records are updated in the Delta table.

In [0]:
# Read the updated Delta table
updated_product_df = spark.read.format("delta").table("Product")

# Show the updated data
print("Updated Product Delta table content:")
updated_product_df.show()

# You can also perform specific checks, e.g., count of records or check specific updated values
# Example: Count total records
print(f"Total records in Product Delta table: {updated_product_df.count()}")

# Example: Check if a specific updated record exists and has the new value
# (Requires knowledge of a specific change in Product_data.csv)
# updated_product_df.filter("ProductID = 'P001'").show()